# Imports

In [8]:
import optuna
import pickle

import numpy as np
import pandas as pd

from utils import MulticlassThresholdOptimizer, load_pickle
from lightgbm import LGBMClassifier

from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold, cross_val_predict

## Utils

In [9]:
label_encoder = load_pickle('../models/label_encoder.pkl')

# Loading Datasets

In [10]:
X_train = pd.read_parquet('../data/X_train_stacking_layer_one.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

X_test = pd.read_parquet('../data/X_test_stacking_layer_one.parquet')

In [11]:
X_train.head()

,lgbm_0,lgbm_1,cat_0,cat_1,xgb_0,xgb_1,hist_0,hist_1,extra_0,extra_1,rf_0,rf_1
0,9.999566e-01,0.000043,0.999900,0.000050,0.999734,0.000192,9.998896e-01,0.000105,0.912643,0.007039,0.999665,0.000155
1,9.895666e-01,0.000366,0.986604,0.000093,0.985616,0.000664,9.819315e-01,0.000384,0.803503,0.004824,0.965267,0.000184
2,3.890271e-07,1.000000,0.000016,0.999984,0.000088,0.999889,3.930889e-07,1.000000,0.006700,0.965519,0.000113,0.999887
3,9.998353e-01,0.000164,0.999925,0.000071,0.999614,0.000265,9.991001e-01,0.000895,0.932451,0.006030,0.999310,0.000345
4,9.993004e-01,0.000692,0.999187,0.000766,0.998981,0.000832,9.846569e-01,0.015320,0.915246,0.008021,0.997708,0.000696


In [12]:
X_test.head()

,lgbm_0,lgbm_1,cat_0,cat_1,xgb_0,xgb_1,hist_0,hist_1,extra_0,extra_1,rf_0,rf_1
0,0.999271,0.000708,0.996756,0.002437,0.997384,0.002024,0.999265,0.000701,0.611632,0.076394,0.977833,0.009356
1,0.997605,0.002393,0.997894,0.002102,0.998017,0.001866,0.971958,0.028034,0.942311,0.018000,0.998781,0.000921
2,0.998315,0.000205,0.999515,0.000035,0.994165,0.001013,0.996011,0.000573,0.530537,0.015231,0.980001,0.002162
3,0.000593,0.000201,0.001054,0.000337,0.003027,0.001288,0.000174,0.000112,0.086899,0.066500,0.009974,0.007596
4,0.999900,0.000097,0.999687,0.000311,0.999482,0.000408,0.999597,0.000382,0.945084,0.015204,0.999542,0.000244


# Machine Learning

In [14]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train_fold = X.iloc[train_idx, :]
        X_valid_fold = X.iloc[valid_idx, :]

        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        model = LGBMClassifier(
            objective='multiclass',
            metric='multi_logloss',
            boosting_type='gbdt',
            verbosity=-1,
            n_estimators=2000,
            random_state=42,
            n_jobs=1,
            num_leaves=trial.suggest_int('num_leaves', 16, 256),
            max_depth=trial.suggest_int('max_depth', 3, 12),
            learning_rate=trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            lambda_l1=trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
            lambda_l2=trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
            feature_fraction=trial.suggest_float('feature_fraction', 0.6, 1.0),
            bagging_fraction=trial.suggest_float('bagging_fraction', 0.6, 1.0),
            bagging_freq=trial.suggest_int('bagging_freq', 1, 7),
            min_child_samples=trial.suggest_int('min_child_samples', 10, 100),
        ).fit(X_train_fold, y_train_fold)

        proba = model.predict_proba(X_valid_fold)

        score = log_loss(y_valid_fold, proba)
        scores.append(score)

        trial.report(np.mean(scores), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(scores)


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train.class_encoded), n_trials=10, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-06-04 16:58:29,890] A new study created in memory with name: no-name-f8bc9f7d-4239-44b8-9d03-1455126fa049
Best trial: 6. Best value: 0.0879368:  10%|█████████████▍                                                                                                                        | 1/10 [20:45<3:06:45, 1245.07s/it]

[I 2026-06-04 17:19:14,954] Trial 6 finished with value: 0.08793677481885516 and parameters: {'num_leaves': 150, 'max_depth': 3, 'learning_rate': 0.028445903054592842, 'lambda_l1': 7.298390485745166, 'lambda_l2': 0.0015949844187547971, 'feature_fraction': 0.869484007666258, 'bagging_fraction': 0.9361272305818251, 'bagging_freq': 1, 'min_child_samples': 92}. Best is trial 6 with value: 0.08793677481885516.


Best trial: 2. Best value: 0.0890054:  20%|███████████████████████████                                                                                                            | 2/10 [34:27<2:12:50, 996.34s/it]

[I 2026-06-04 17:32:57,179] Trial 2 finished with value: 0.08900544263676981 and parameters: {'num_leaves': 22, 'max_depth': 7, 'learning_rate': 0.027583256890709597, 'lambda_l1': 0.5917661656604233, 'lambda_l2': 0.03436881503142655, 'feature_fraction': 0.7142980051535769, 'bagging_fraction': 0.9642431234755495, 'bagging_freq': 4, 'min_child_samples': 100}. Best is trial 2 with value: 0.08900544263676981.


Best trial: 8. Best value: 0.0919241:  30%|████████████████████████████████████████▌                                                                                              | 3/10 [40:39<1:23:00, 711.51s/it]

...

[I 2026-06-04 17:57:24,602] Trial 5 finished with value: 0.1433964729642635 and parameters: {'num_leaves': 190, 'max_depth': 8, 'learning_rate': 0.14548715343599952, 'lambda_l1': 0.02788976790692577, 'lambda_l2': 0.5773241377812052, 'feature_fraction': 0.7313879440338058, 'bagging_fraction': 0.6873752199275773, 'bagging_freq': 4, 'min_child_samples': 43}. Best is trial 5 with value: 0.1433964729642635.


Best trial: 5. Best value: 0.143396:  90%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍             | 9/10 [1:06:24<05:17, 317.39s/it]

[I 2026-06-04 18:04:54,726] Trial 1 finished with value: 0.11976288459179843 and parameters: {'num_leaves': 152, 'max_depth': 11, 'learning_rate': 0.15690756142570894, 'lambda_l1': 2.5508181888516055, 'lambda_l2': 0.0561507502410961, 'feature_fraction': 0.7874965293523489, 'bagging_fraction': 0.7770666193024611, 'bagging_freq': 4, 'min_child_samples': 50}. Best is trial 5 with value: 0.1433964729642635.


Best trial: 5. Best value: 0.143396: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [1:09:34<00:00, 417.50s/it]

[I 2026-06-04 18:08:04,868] Trial 9 finished with value: 0.09929548985553469 and parameters: {'num_leaves': 256, 'max_depth': 10, 'learning_rate': 0.035598815609657886, 'lambda_l1': 2.1675150782899153, 'lambda_l2': 1.0532907067825874, 'feature_fraction': 0.6591596955765096, 'bagging_fraction': 0.7146902029457115, 'bagging_freq': 5, 'min_child_samples': 71}. Best is trial 5 with value: 0.1433964729642635.
Best trial score:
0.1433964729642635

Best params:
{'num_leaves': 190, 'max_depth': 8, 'learning_rate': 0.14548715343599952, 'lambda_l1': 0.02788976790692577, 'lambda_l2': 0.5773241377812052, 'feature_fraction': 0.7313879440338058, 'bagging_fraction': 0.6873752199275773, 'bagging_freq': 4, 'min_child_samples': 43}


In [51]:
study.optimize(lambda trial: objective(trial, X_train, y_train.class_encoded), n_trials=10, n_jobs=-1, show_progress_bar=True)

  0%|                                                                                                                                                                                        | 0/10 [10:12<?, ?it/s]


KeyboardInterrupt: 

In [43]:
best_params = {
    'num_leaves': 150, 
    'max_depth': 3, 
    'learning_rate': 0.028445903054592842, 
    'lambda_l1': 7.298390485745166, 
    'lambda_l2': 0.0015949844187547971, 
    'feature_fraction': 0.869484007666258, 
    'bagging_fraction': 0.9361272305818251, 
    'bagging_freq': 1, 
    'min_child_samples': 92
}


lgbm = LGBMClassifier(
    **best_params,
    objective='multiclass',
    metric='multi_logloss',
    boosting_type='gbdt',
    verbosity=-1,
    n_estimators=2000,
    random_state=42,
    n_jobs=1,
).fit(X_train, y_train.class_encoded)

train_proba = cross_val_predict(lgbm, X_train, y_train.class_encoded, cv=StratifiedKFold(shuffle=True, random_state=42, n_splits=5), n_jobs=-1, method='predict_proba')

In [44]:
test_proba = lgbm.predict_proba(X_test)

In [45]:
optimizer = MulticlassThresholdOptimizer()
optimizer.fit(train_proba, y_train.class_encoded)

,n_splits,5
,method,'Nelder-Mead'
,maxiter,500
,random_state,42


In [46]:
# test_pred = optimizer.predict(test_proba)
# sub_labels = label_encoder.inverse_transform(test_pred)

In [47]:
final_predictions_idx = optimizer.predict(test_proba)

class_mapping = {0: 'GALAXY', 1: 'QSO', 2: 'STAR'}
sub_labels = [class_mapping[idx] for idx in final_predictions_idx]

# Submission

In [48]:
submission = pd.read_csv('../data/sample_submission.csv')
submission['class'] = sub_labels

submission.to_csv('../data/submission_stacking_lgbm.csv', index=False)

In [49]:
submission.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [50]:
X_train.columns

Index(['lgbm_0', 'lgbm_1', 'cat_0', 'cat_1', 'xgb_0', 'xgb_1', 'hist_0',
       'hist_1', 'extra_0', 'extra_1', 'rf_0', 'rf_1'],
      dtype='str')